In [1]:
import pandas as pd
import re
import math

# Stop words
stop_words = {"the","is","and","to","in","of","a","on","for","with"}

# CSV-ից կարդալ տեքստերը և կատեգորիաները
df = pd.read_csv("bbc-text.csv", encoding="latin1")  # assuming columns "text" and "category"
docs = df["text"].tolist()
cats = df["category"].tolist()

# Exercise 1: Preprocess
tokens = []
for text in docs:
    words = re.findall(r"[a-z]+", text.lower())
    words = [w for w in words if w not in stop_words]
    tokens.append(words)

# Exercise 2: Vocabulary
freq = {}
for doc in tokens:
    for w in doc:
        freq[w] = freq.get(w,0)+1
vocab = {w:i for i,w in enumerate(freq.keys())}

# Exercise 3: Bag of Words
X = []
for doc in tokens:
    vec = [0]*len(vocab)
    for w in doc:
        if w in vocab:
            vec[vocab[w]] += 1
    X.append(vec)

# Exercise 4: Term Frequency
TF = []
for vec in X:
    s = sum(vec) or 1
    TF.append([c/s for c in vec])

# Exercise 5: Inverse Document Frequency
N = len(X)
df_count = [0]*len(vocab)
for vec in X:
    for j,c in enumerate(vec):
        if c>0: df_count[j]+=1
IDF = [math.log((N+1)/(d+1))+1 for d in df_count]

# Exercise 6: TF-IDF
TFIDF = []
for tf in TF:
    TFIDF.append([a*b for a,b in zip(tf,IDF)])

# Exercise 7: Centroids
labels = list(set(cats))
centroids = {lab:[0]*len(vocab) for lab in labels}
counts = {lab:0 for lab in labels}
for vec,lab in zip(TFIDF,cats):
    centroids[lab] = [a+b for a,b in zip(centroids[lab],vec)]
    counts[lab]+=1
for lab in labels:
    c = counts[lab] or 1
    centroids[lab] = [v/c for v in centroids[lab]]

# Exercise 8: Prediction
preds = []
for vec in TFIDF:
    best_lab=None; best_sim=-1
    for lab in labels:
        num=sum(x*y for x,y in zip(vec,centroids[lab]))
        den=(sum(x*x for x in vec)**0.5)*(sum(y*y for y in centroids[lab])**0.5)
        sim=num/den if den else 0
        if sim>best_sim:
            best_sim=sim; best_lab=lab
    preds.append(best_lab)

# Exercise 9: Results
print("True labels:     ",cats)
print("Predicted labels:",preds)

True labels:      ['tech', 'business', 'sport', 'sport', 'entertainment', 'politics', 'politics', 'sport', 'sport', 'entertainment', 'entertainment', 'business', 'business', 'politics', 'sport', 'business', 'politics', 'sport', 'business', 'tech', 'tech', 'tech', 'sport', 'sport', 'tech', 'sport', 'entertainment', 'tech', 'politics', 'entertainment', 'politics', 'tech', 'entertainment', 'entertainment', 'business', 'politics', 'tech', 'entertainment', 'politics', 'business', 'politics', 'sport', 'business', 'sport', 'tech', 'entertainment', 'politics', 'politics', 'politics', 'business', 'sport', 'politics', 'business', 'business', 'sport', 'politics', 'business', 'sport', 'sport', 'business', 'business', 'sport', 'business', 'sport', 'business', 'tech', 'business', 'entertainment', 'tech', 'business', 'politics', 'business', 'politics', 'sport', 'business', 'tech', 'business', 'sport', 'sport', 'business', 'business', 'sport', 'politics', 'business', 'entertainment', 'politics', 'poli